# 05: Sağlam değerlendirme (VisA cashew)

Eşik seçimi ayrı bir doğrulama kümesinden, sonuçlar birden fazla rastgele tohumla.
Eğitim: train/good'un %80'i (360), doğrulama: %20 (90, model görmedi), test: 50 sağlam + 100 kusurlu.
Veri seti: VisA (Zou et al., ECCV 2022).

In [12]:
!pip install anomalib

In [19]:
from anomalib.data import Visa
dm = Visa(root="/kaggle/working/datasets/visa", category="cashew")
dm.prepare_data()

In [20]:
!ls /kaggle/working/datasets/visa/visa_pytorch/cashew

ground_truth  test  train


## 1. Eşik doğrulamadan, 5 tohum
Eşik = doğrulama skorlarının 95. yüzdeliği. Eğitim/doğrulama bölmesi ve PatchCore coreset örneklemesi tohuma göre değişir.

| Metrik | Ortalama ± sapma (5 tohum) |
|---|---|
| Image AUROC | 0.961 ± 0.004 |
| Precision | 0.985 ± 0.019 |
| Recall | 0.718 ± 0.065 |
| Yanlış alarm | 0.024 ± 0.033 |
| F1 | 0.829 ± 0.030 |

Sıralama kalitesi (AUROC) kararlı, eşik ve recall tohumlara göre belirgin oynuyor (eşik 0.55-0.64, recall 0.64-0.80).
Sapma test görüntülerinin seçiminden gelen belirsizliği içermez.

In [23]:
%%writefile temiz_degerlendir.py
import sys, json, random, os, glob, shutil
from pathlib import Path
import torch
from anomalib.data import Folder
from anomalib.models import Patchcore
from anomalib.engine import Engine

seed = int(sys.argv[1]) if len(sys.argv) > 1 else 0
SRC = Path("/kaggle/working/datasets/visa/visa_pytorch/cashew")
DST = Path(f"/kaggle/working/cashew_split_s{seed}")

def bagla(kaynak, hedef):
    try:
        os.link(kaynak, hedef)
    except OSError:
        shutil.copy2(kaynak, hedef)

# 1) Eğitim / doğrulama ayrımı
if DST.exists():
    shutil.rmtree(DST)
(DST/"train_good").mkdir(parents=True)
(DST/"val_good").mkdir(parents=True)
tum = sorted(glob.glob(str(SRC/"train/good/*")))
random.Random(seed).shuffle(tum)
k = int(len(tum) * 0.8)
egit, val = tum[:k], tum[k:]
for f in egit:
    bagla(f, str(DST/"train_good"/Path(f).name))
for f in val:
    bagla(f, str(DST/"val_good"/Path(f).name))
print("SONUC eğitim:", len(egit), "doğrulama:", len(val), flush=True)

# 2) Sadece eğitim kümesiyle PatchCore (maske gerektirmeyen Folder)
dm = Folder(name="cashew",
            normal_dir=str(DST/"train_good"),
            normal_test_dir=str(SRC/"test/good"),
            abnormal_dir=str(SRC/"test/bad"),
            train_batch_size=16, eval_batch_size=16, num_workers=2)
model = Patchcore()
engine = Engine(default_root_dir=f"/kaggle/working/results/temiz_s{seed}")
engine.fit(datamodule=dm, model=model)

# 3) Üç kümenin skorları
def skorla(klasor):
    tahm = engine.predict(model=model, data_path=str(klasor))
    return [float(s) for s in torch.cat([p.pred_score.flatten().cpu() for p in tahm])]

sonuc = {
    "seed": seed,
    "n_egitim": len(egit), "n_val": len(val),
    "val": skorla(DST/"val_good"),
    "test_good": skorla(SRC/"test/good"),
    "test_bad": skorla(SRC/"test/bad"),
}
json.dump(sonuc, open(f"/kaggle/working/skorlar_cashew_s{seed}.json", "w"))
print("SONUC skor sayıları: val", len(sonuc["val"]), "test_good", len(sonuc["test_good"]),
      "test_bad", len(sonuc["test_bad"]), flush=True)

Overwriting temiz_degerlendir.py


In [24]:
!python temiz_degerlendir.py 0 2>&1 | grep -E "SONUC|Error|Traceback"

SONUC eğitim: 360 doğrulama: 90
SONUC skor sayıları: val 90 test_good 50 test_bad 100


In [25]:
import json, numpy as np
from sklearn.metrics import roc_auc_score

d = json.load(open("/kaggle/working/skorlar_cashew_s0.json"))
val, g, b = np.array(d["val"]), np.array(d["test_good"]), np.array(d["test_bad"])

esik = float(np.percentile(val, 95))      # eşik SADECE doğrulamadan
auc  = roc_auc_score([0]*len(g) + [1]*len(b), np.concatenate([g, b]))

fp, tn = int((g > esik).sum()), int((g <= esik).sum())
tp, fn = int((b > esik).sum()), int((b <= esik).sum())
precision = tp / (tp + fp) if tp + fp else 0
recall    = tp / (tp + fn)
fpr       = fp / (fp + tn)
f1        = 2*precision*recall / (precision + recall) if precision + recall else 0

print(f"eşik (doğrulama %95): {esik:.3f}")
print(f"AUROC {auc:.3f} | precision {precision:.3f} | recall {recall:.3f} | yanlış alarm {fpr:.3f} | F1 {f1:.3f}")
print(f"doğrulama skor aralığı {val.min():.3f}-{val.max():.3f} | test sağlam {g.min():.3f}-{g.max():.3f}")

eşik (doğrulama %95): 0.550
AUROC 0.965 | precision 0.952 | recall 0.800 | yanlış alarm 0.080 | F1 0.870
doğrulama skor aralığı 0.399-0.730 | test sağlam 0.404-0.570


In [26]:
!for s in 1 2 3 4; do python temiz_degerlendir.py $s 2>&1 | grep -E "SONUC|Traceback|Error"; done

SONUC eğitim: 360 doğrulama: 90
SONUC skor sayıları: val 90 test_good 50 test_bad 100
SONUC eğitim: 360 doğrulama: 90
SONUC skor sayıları: val 90 test_good 50 test_bad 100
SONUC eğitim: 360 doğrulama: 90
SONUC skor sayıları: val 90 test_good 50 test_bad 100
SONUC eğitim: 360 doğrulama: 90
SONUC skor sayıları: val 90 test_good 50 test_bad 100


## 2. Üst sınır (eşik test sağlamlarından seçilirse)
Dağıtılabilir bir yöntem değil, yalnızca modelin tavanını gösterir.

| Yanlış alarm hedefi | Recall (5 tohum aralığı) |
|---|---|
| %2 | 0.76-0.80 |
| %5 | 0.78-0.80 |
| %10 | 0.83-0.90 |

In [27]:
import json, numpy as np
from sklearn.metrics import roc_auc_score

satirlar = []
for s in range(5):
    d = json.load(open(f"/kaggle/working/skorlar_cashew_s{s}.json"))
    val, g, b = np.array(d["val"]), np.array(d["test_good"]), np.array(d["test_bad"])
    e = float(np.percentile(val, 95))
    auc = roc_auc_score([0]*len(g) + [1]*len(b), np.concatenate([g, b]))
    fp, tp = int((g > e).sum()), int((b > e).sum())
    prec = tp / (tp + fp) if tp + fp else 0
    rec, fpr = tp / len(b), fp / len(g)
    f1 = 2*prec*rec/(prec+rec) if prec+rec else 0
    satirlar.append([auc, prec, rec, fpr, f1, e])
    print(f"seed {s}: eşik {e:.3f} AUROC {auc:.3f} precision {prec:.3f} recall {rec:.3f} yanlış alarm {fpr:.3f} F1 {f1:.3f}")

a = np.array(satirlar)
for ad, i in [("AUROC", 0), ("precision", 1), ("recall", 2), ("yanlış alarm", 3), ("F1", 4)]:
    print(f"ORTALAMA {ad}: {a[:, i].mean():.3f} ± {a[:, i].std(ddof=1):.3f}")

seed 0: eşik 0.550 AUROC 0.965 precision 0.952 recall 0.800 yanlış alarm 0.080 F1 0.870
seed 1: eşik 0.642 AUROC 0.960 precision 1.000 recall 0.640 yanlış alarm 0.000 F1 0.780
seed 2: eşik 0.596 AUROC 0.959 precision 0.987 recall 0.770 yanlış alarm 0.020 F1 0.865
seed 3: eşik 0.611 AUROC 0.957 precision 0.986 recall 0.690 yanlış alarm 0.020 F1 0.812
seed 4: eşik 0.638 AUROC 0.965 precision 1.000 recall 0.690 yanlış alarm 0.000 F1 0.817
ORTALAMA AUROC: 0.961 ± 0.004
ORTALAMA precision: 0.985 ± 0.019
ORTALAMA recall: 0.718 ± 0.065
ORTALAMA yanlış alarm: 0.024 ± 0.033
ORTALAMA F1: 0.829 ± 0.038


In [28]:
for s in range(5):
    d = json.load(open(f"/kaggle/working/skorlar_cashew_s{s}.json"))
    g, b = np.array(d["test_good"]), np.array(d["test_bad"])
    for hedef in [0.02, 0.05, 0.10]:
        e = float(np.quantile(g, 1 - hedef))
        print(f"seed {s} yanlış alarm hedefi {hedef:.2f}: recall {(b > e).mean():.2f}")

seed 0 yanlış alarm hedefi 0.02: recall 0.80
seed 0 yanlış alarm hedefi 0.05: recall 0.80
seed 0 yanlış alarm hedefi 0.10: recall 0.90
seed 1 yanlış alarm hedefi 0.02: recall 0.78
seed 1 yanlış alarm hedefi 0.05: recall 0.80
seed 1 yanlış alarm hedefi 0.10: recall 0.85
seed 2 yanlış alarm hedefi 0.02: recall 0.78
seed 2 yanlış alarm hedefi 0.05: recall 0.78
seed 2 yanlış alarm hedefi 0.10: recall 0.83
seed 3 yanlış alarm hedefi 0.02: recall 0.76
seed 3 yanlış alarm hedefi 0.05: recall 0.78
seed 3 yanlış alarm hedefi 0.10: recall 0.88
seed 4 yanlış alarm hedefi 0.02: recall 0.78
seed 4 yanlış alarm hedefi 0.05: recall 0.80
seed 4 yanlış alarm hedefi 0.10: recall 0.89


In [29]:
%%writefile isik_temiz.py
import sys, json, random, os, glob, shutil
from pathlib import Path
import cv2, numpy as np, torch
from anomalib.data import Folder
from anomalib.models import Patchcore
from anomalib.engine import Engine

seed = int(sys.argv[1])
SRC = Path("/kaggle/working/datasets/visa/visa_pytorch/cashew")
DST = Path(f"/kaggle/working/cashew_split_s{seed}")
IMG = Path("/kaggle/working/isik_imgs")
CARPANLAR = [0.5, 0.75, 1.0, 1.25, 1.5]

def bagla(kaynak, hedef):
    try:
        os.link(kaynak, hedef)
    except OSError:
        shutil.copy2(kaynak, hedef)

# Aynı bölme: temiz_degerlendir.py ile aynı seed, aynı sıra
if DST.exists():
    shutil.rmtree(DST)
(DST/"train_good").mkdir(parents=True)
(DST/"val_good").mkdir(parents=True)
tum = sorted(glob.glob(str(SRC/"train/good/*")))
random.Random(seed).shuffle(tum)
k = int(len(tum) * 0.8)
egit, val = tum[:k], tum[k:]
for f in egit:
    bagla(f, str(DST/"train_good"/Path(f).name))
for f in val:
    bagla(f, str(DST/"val_good"/Path(f).name))

dm = Folder(name="cashew", normal_dir=str(DST/"train_good"),
            normal_test_dir=str(SRC/"test/good"), abnormal_dir=str(SRC/"test/bad"),
            train_batch_size=16, eval_batch_size=16, num_workers=2)
model = Patchcore()
engine = Engine(default_root_dir=f"/kaggle/working/results/isik_s{seed}")
engine.fit(datamodule=dm, model=model)

def skorla(klasor):
    tahm = engine.predict(model=model, data_path=str(klasor))
    return [float(s) for s in torch.cat([p.pred_score.flatten().cpu() for p in tahm])]

def parlat(liste, ad, c):
    hedef = IMG/f"{ad}_{c}"
    if hedef.exists() and len(list(hedef.glob("*.png"))) == len(liste):
        return hedef
    hedef.mkdir(parents=True, exist_ok=True)
    for i, f in enumerate(liste):
        im = cv2.imread(f).astype(np.float32) * c
        cv2.imwrite(str(hedef/f"{i:04d}.png"), np.clip(im, 0, 255).astype(np.uint8))
    return hedef

iyi = sorted(glob.glob(str(SRC/"test/good/*")))
kotu = sorted(glob.glob(str(SRC/"test/bad/*")))
sonuc = {"seed": seed, "val": skorla(DST/"val_good"), "good": {}, "bad": {}}
for c in CARPANLAR:
    if c == 1.0:
        sonuc["good"][str(c)] = skorla(SRC/"test/good")
        sonuc["bad"][str(c)] = skorla(SRC/"test/bad")
    else:
        sonuc["good"][str(c)] = skorla(parlat(iyi, "good", c))
        sonuc["bad"][str(c)] = skorla(parlat(kotu, "bad", c))
    print("SONUC seed", seed, "carpan", c, "bitti", flush=True)
json.dump(sonuc, open(f"/kaggle/working/isik_skorlar_s{seed}.json", "w"))

Writing isik_temiz.py


## 3. Simüle ışık testi (3 tohum: 0, 1, 4)
Eşik normal ışıkta doğrulamadan seçildi ve sabit tutuldu. Parlaklık düzgün çarpanla değiştirildi (gerçek ışık değişimi değil).

| Çarpan | AUROC | Yanlış alarm | Recall |
|---|---|---|---|
| x0.5 | 0.965 ± 0.003 | 0.240 ± 0.131 | 0.940 ± 0.072 |
| x0.75 | 0.964 ± 0.003 | 0.033 ± 0.042 | 0.747 ± 0.081 |
| x1.0 | 0.960 ± 0.003 | 0.013 ± 0.023 | 0.713 ± 0.080 |
| x1.25 | 0.968 ± 0.003 | 0.173 ± 0.103 | 0.910 ± 0.056 |
| x1.5 | 0.953 ± 0.004 | 0.947 ± 0.076 | 1.000 ± 0.000 |

Recall 1.0 ile yanlış alarm 0.95 birlikte: sistem her şeye "kusurlu" diyor. Tohumlar arası sapma yüksek (3 tohum).

In [30]:
!for s in 0 1 2; do python isik_temiz.py $s 2>&1 | grep -E "SONUC|Traceback|Error"; done

SONUC seed 4 carpan 0.5 bitti
SONUC seed 4 carpan 0.75 bitti
SONUC seed 4 carpan 1.0 bitti
SONUC seed 4 carpan 1.25 bitti
SONUC seed 4 carpan 1.5 bitti
SONUC seed 4 carpan 0.5 bitti
SONUC seed 4 carpan 0.75 bitti
SONUC seed 4 carpan 1.0 bitti
SONUC seed 4 carpan 1.25 bitti
SONUC seed 4 carpan 1.5 bitti
SONUC seed 4 carpan 0.5 bitti
SONUC seed 4 carpan 0.75 bitti
SONUC seed 4 carpan 1.0 bitti
SONUC seed 4 carpan 1.25 bitti
SONUC seed 4 carpan 1.5 bitti


In [32]:
!python isik_temiz.py 0 2>&1 | grep -E "SONUC|Traceback|Error"
!python isik_temiz.py 1 2>&1 | grep -E "SONUC|Traceback|Error"

SONUC seed 0 carpan 0.5 bitti
SONUC seed 0 carpan 0.75 bitti
SONUC seed 0 carpan 1.0 bitti
SONUC seed 0 carpan 1.25 bitti
SONUC seed 0 carpan 1.5 bitti
SONUC seed 1 carpan 0.5 bitti
SONUC seed 1 carpan 0.75 bitti
SONUC seed 1 carpan 1.0 bitti
SONUC seed 1 carpan 1.25 bitti
SONUC seed 1 carpan 1.5 bitti


## 4. Pixel AUROC (3 tohum)
Isı haritası ile VisA maskeleri karşılaştırıldı (maske harita çözünürlüğüne küçültüldü, sağlam görüntüler boş maskeyle dahil).
Pixel AUROC ≈ 0.991 (0.9907-0.991). Kusurlu piksel oranı %1.1: sınıf dengesizliği nedeniyle yüksek sayı küçük kusurların kaçırıldığını gizleyebilir.

In [33]:
import json, numpy as np
from sklearn.metrics import roc_auc_score

CARPANLAR = ["0.5", "0.75", "1.0", "1.25", "1.5"]
SEEDLER = [0, 1, 4]
tablo = {c: [] for c in CARPANLAR}
for sd in SEEDLER:
    d = json.load(open(f"/kaggle/working/isik_skorlar_s{sd}.json"))
    esik = float(np.percentile(d["val"], 95))      # eşik yalnızca normal ışıkta, doğrulamadan
    for c in CARPANLAR:
        g, b = np.array(d["good"][c]), np.array(d["bad"][c])
        auc = roc_auc_score([0]*len(g) + [1]*len(b), np.concatenate([g, b]))
        tablo[c].append([auc, (g > esik).mean(), (b > esik).mean()])

for c in CARPANLAR:
    a = np.array(tablo[c])
    print(f"x{c}: AUROC {a[:,0].mean():.3f}±{a[:,0].std(ddof=1):.3f} | yanlış alarm {a[:,1].mean():.3f}±{a[:,1].std(ddof=1):.3f} | recall {a[:,2].mean():.3f}±{a[:,2].std(ddof=1):.3f}")

x0.5: AUROC 0.965±0.003 | yanlış alarm 0.240±0.131 | recall 0.940±0.072
x0.75: AUROC 0.964±0.003 | yanlış alarm 0.033±0.042 | recall 0.747±0.081
x1.0: AUROC 0.960±0.003 | yanlış alarm 0.013±0.023 | recall 0.713±0.080
x1.25: AUROC 0.968±0.003 | yanlış alarm 0.173±0.103 | recall 0.910±0.056
x1.5: AUROC 0.953±0.004 | yanlış alarm 0.947±0.076 | recall 1.000±0.000


In [34]:
!ls /kaggle/working/datasets/visa/visa_pytorch/cashew/ground_truth
!ls /kaggle/working/datasets/visa/visa_pytorch/cashew/ground_truth/bad | head -3
!ls /kaggle/working/datasets/visa/visa_pytorch/cashew/test/bad | head -3

bad
000.png
001.png
002.png
000.JPG
001.JPG
002.JPG


In [35]:
%%writefile pixel_auroc.py
import sys, json, random, os, glob, shutil
from pathlib import Path
import cv2, numpy as np, torch
from sklearn.metrics import roc_auc_score
from anomalib.data import Folder
from anomalib.models import Patchcore
from anomalib.engine import Engine

seed = int(sys.argv[1])
SRC = Path("/kaggle/working/datasets/visa/visa_pytorch/cashew")
DST = Path(f"/kaggle/working/cashew_split_s{seed}")

def bagla(kaynak, hedef):
    try:
        os.link(kaynak, hedef)
    except OSError:
        shutil.copy2(kaynak, hedef)

if DST.exists():
    shutil.rmtree(DST)
(DST/"train_good").mkdir(parents=True)
tum = sorted(glob.glob(str(SRC/"train/good/*")))
random.Random(seed).shuffle(tum)
k = int(len(tum) * 0.8)
for f in tum[:k]:
    bagla(f, str(DST/"train_good"/Path(f).name))

dm = Folder(name="cashew", normal_dir=str(DST/"train_good"),
            normal_test_dir=str(SRC/"test/good"), abnormal_dir=str(SRC/"test/bad"),
            train_batch_size=16, eval_batch_size=16, num_workers=2)
model = Patchcore()
engine = Engine(default_root_dir=f"/kaggle/working/results/pixel_s{seed}")
engine.fit(datamodule=dm, model=model)

def haritalar(klasor):
    tahm = engine.predict(model=model, data_path=str(klasor))
    out = []
    for b in tahm:
        for yol, h in zip(b.image_path, b.anomaly_map):
            out.append((Path(yol), h.squeeze().cpu().numpy()))
    return out

tum_p, tum_y = [], []
eslesen = eksik = 0
for ad, klasor in [("good", SRC/"test/good"), ("bad", SRC/"test/bad")]:
    for yol, h in haritalar(klasor):
        if ad == "good":
            maske = np.zeros(h.shape, dtype=np.uint8)
        else:
            adaylar = glob.glob(str(SRC/"ground_truth/bad"/(yol.stem + ".*")))
            if not adaylar:
                eksik += 1
                continue
            m = cv2.imread(adaylar[0], cv2.IMREAD_GRAYSCALE)
            m = cv2.resize(m, (h.shape[1], h.shape[0]), interpolation=cv2.INTER_NEAREST)
            maske = (m > 127).astype(np.uint8)
            eslesen += 1
        tum_p.append(h.flatten()); tum_y.append(maske.flatten())

print("SONUC maske eşleşen:", eslesen, "eksik:", eksik, flush=True)
p = np.concatenate(tum_p); y = np.concatenate(tum_y)
print("SONUC kusurlu piksel oranı:", round(float(y.mean()), 4), flush=True)
print("SONUC seed", seed, "pixel AUROC:", round(float(roc_auc_score(y, p)), 4), flush=True)
json.dump({"seed": seed, "pixel_auroc": float(roc_auc_score(y, p)),
           "eslesen": eslesen, "eksik": eksik},
          open(f"/kaggle/working/pixel_s{seed}.json", "w"))

Writing pixel_auroc.py


In [36]:
!python pixel_auroc.py 0 2>&1 | grep -E "SONUC|Traceback|Error"

SONUC maske eşleşen: 100 eksik: 0
SONUC kusurlu piksel oranı: 0.0113
SONUC seed 0 pixel AUROC: 0.991


In [37]:
!python pixel_auroc.py 1 2>&1 | grep -E "SONUC|Traceback|Error"
!python pixel_auroc.py 2 2>&1 | grep -E "SONUC|Traceback|Error"

SONUC maske eşleşen: 100 eksik: 0
SONUC kusurlu piksel oranı: 0.0113
SONUC seed 1 pixel AUROC: 0.9907
SONUC maske eşleşen: 100 eksik: 0
SONUC kusurlu piksel oranı: 0.0113
SONUC seed 2 pixel AUROC: 0.9908


In [38]:
import json, numpy as np
v = [json.load(open(f"/kaggle/working/pixel_s{sd}.json"))["pixel_auroc"] for sd in [0, 1, 2]]
print(f"pixel AUROC: {np.mean(v):.4f} ± {np.std(v, ddof=1):.4f}  ({[round(x, 4) for x in v]})")

pixel AUROC: 0.9908 ± 0.0001  ([0.991, 0.9907, 0.9908])


## 5. engine.test ile elle hesap
Temiz oturumda ikisi de 0.9664 verdi (75 görüntülük alt küme: `Folder` test klasörünün %50'sini doğrulamaya ayırıyor).
MVTec `metal_nut` üzerindeki eski 0.5 sonucu yeniden üretilemedi, büyük olasılıkla çöken oturumdan kalan bozuk durumdu (kanıtlanmadı).

In [39]:
%%writefile test_karsilastir.py
import os, glob, random, shutil
from pathlib import Path
import torch
from sklearn.metrics import roc_auc_score
from anomalib.data import Folder
from anomalib.models import Patchcore
from anomalib.engine import Engine

SRC = Path("/kaggle/working/datasets/visa/visa_pytorch/cashew")
DST = Path("/kaggle/working/cashew_split_s0")   # temiz_degerlendir.py bölmesi mevcut
if not (DST/"train_good").exists():
    raise SystemExit("HATA: önce temiz_degerlendir.py 0 çalışmalı")

dm = Folder(name="cashew", normal_dir=str(DST/"train_good"),
            normal_test_dir=str(SRC/"test/good"), abnormal_dir=str(SRC/"test/bad"),
            train_batch_size=16, eval_batch_size=16, num_workers=2)
model = Patchcore()
engine = Engine(default_root_dir="/kaggle/working/results/test_karsilastir")
engine.fit(datamodule=dm, model=model)

# 1) Elle: predict + roc_auc_score
tahm = engine.predict(model=model, dataloaders=dm.test_dataloader())
skor = torch.cat([p.pred_score.flatten().cpu() for p in tahm]).numpy()
etik = torch.cat([p.gt_label.flatten().cpu() for p in tahm]).numpy()
print("SONUC elle AUROC:", round(float(roc_auc_score(etik, skor)), 4),
      "| görüntü:", len(skor), "kusurlu:", int(etik.sum()), flush=True)

# 2) anomalib'in engine.test sonucu
sonuc = engine.test(model=model, datamodule=dm)
print("SONUC engine.test:", {k: round(float(v), 4) for k, v in sonuc[0].items()}, flush=True)

Writing test_karsilastir.py


In [40]:
!ls /kaggle/working/cashew_split_s0

train_good


In [41]:
!python test_karsilastir.py 2>&1 | grep -E "SONUC|HATA|Traceback|Error"

SONUC elle AUROC: 0.9664 | görüntü: 75 kusurlu: 50
SONUC engine.test: {'image_AUROC': 0.9664, 'image_F1Score': 0.8936}


In [42]:
!grep -n "val_split" /usr/local/lib/python3.13/dist-packages/anomalib/data/datamodules/image/folder.py | head

82:        val_split_mode (ValSplitMode): Method to obtain validation subset.
84:        val_split_ratio (float): Fraction of images for validation.
134:        val_split_mode: ValSplitMode | str = ValSplitMode.FROM_TEST,
135:        val_split_ratio: float = 0.5,
146:        val_split_mode = ValSplitMode(val_split_mode)
157:            val_split_mode=val_split_mode,
158:            val_split_ratio=val_split_ratio,
